In [1]:
####################
# Step 1: Define your model(s) (Dynamic Section)
####################

def laccurve5_dijkstra(self, stateVars, t, scenario_param=None, prev_var_return=None):
    # prev_var_return is required if you want to have a variable with an initial value that then gets updated each iteration
    # The variable must also be included in the outputs
    # TODO ask John about this issue, how common is it to have variables like this

    import numpy as np
    import math

    # Assign Parameter Values #
    a = self.parameters['a']
    b = self.parameters['b']
    b0 = self.parameters['b0']
    c = self.parameters['c']
    L = self.parameters['L']
    KDMIMILK = self.parameters['KDMIMILK']
    KDMImbw = self.parameters['KDMImbw']
    Hcfeed = self.parameters['Hcfeed']
    Hcmilk = self.parameters['Hcmilk']
    KL = self.parameters['KL']
    expL = self.parameters['expL']
    # pregdate = self.parameters['pregdate']
        # pregdate is replaced with scenario_param
    kf1 = self.parameters['kf1']
    kf2 = self.parameters['kf2']
    milk_value = self.parameters['milk_value']

    # Paramaters that require an initial value #
    if t == 0:
            E = self.parameters['E']
    else:
        E = prev_var_return[5]


    # Variables w/ Differential Equation #
    BW = stateVars[0]
    ER = stateVars[1]
    milkcumul = stateVars[2]
    DMIcumul = stateVars[3]
    IOFCcumul = stateVars[4]

    # Model Equations # 
    feed_price = (101 * Hcfeed + 2.7) / 1000

    BFat = ER/9.0       
    # Body fat

    Lmod = 1.0-(1.0-L)/(1.0+(KL/BFat)**expL)         
        # 1 - (1-L) = L?
        # Modifying value of L? Why the adjustment?

    dijkstra_milk = a * np.exp(b * (1 - np.exp(-b0 * t)) / b0 - c * t)
    # John has modified S to give 4% FCM yield per alveolus 

    NEmilk = E**Lmod*dijkstra_milk                         
        # Net energy milk
        # On slide 28 this is equation for FCM yield but L is modified here
        # Is L modified to give a NEmilk as well as a milk yield (milk)

    milk = NEmilk/Hcmilk                
        # Milk production

    DMINRC = (KDMIMILK * milk + KDMImbw * BW**0.75)*(1.0-math.exp(-0.192*(t/7+3.67)))   
        # Slide 28
        # NRC DMI equation

    fdbk = (kf1*t/7+kf2)*(ER-self.iStateVars[1])/self.iStateVars[1]       
    # Ellis 2006
        # Feedback on DMI

    DMI = DMINRC - fdbk                             
        # Dry matter intake     

    NEI = Hcfeed*DMI                                
        # Net energy intake

    NEmaint = 0.096*BW**0.75                        
        # Net energy maintenance

    if t < scenario_param + 190:                          
        NEgest = 0
    else:
        NEgest = (0.00318*(t-scenario_param-190)-0.0352)/0.218
        # Net energy gestation

    NEreqt = NEmaint + dijkstra_milk + NEgest + 5.12*2.0          
        # NE requirement, from NRC

    E = NEI/NEreqt            
        # Energy balance

    # Differential Equations # 
    dERdt = NEI - NEmaint - NEmilk - NEgest
    # Energy requirement (NE)

    if dERdt < 0:
        dBWdt = dERdt/4.92  # 4.92 Mcal/kg in NRC(1988)
    else:
        dBWdt = dERdt/5.12  # 5.12 Mcal/kg for gain
        # Bodyweight gain

    IOFC = (milk * milk_value) - (DMI * feed_price)

    # Format data for return # 
    differential_return = [dBWdt, dERdt, milk, DMI, IOFC]
    local_variables = locals()
    # Store local variables 
    variable_returns = [local_variables.get(variable_name) for variable_name in self.outputs]
    # Create list of variables to return

    return differential_return, variable_returns

functions_to_include = [laccurve5_dijkstra]



In [2]:
####################
# Step 2: Generate cows from dataset
####################

import pandas as pd
import simoolator as moo

# Import cow data 
cow_data_raw = pd.read_csv('../data/cow_import_data.csv')

# Function to convert a .csv to the required format
def cow_wrangler(df):
    # Initialize empty lists to store data for each cow
    cow_ids = []
    iStateVars_list = []
    parameters_dict_list = []

    # Iterate over rows
    for index, row in df.iterrows():
        # Extract cow ID
        cow_id = row['cow_ID']
        cow_ids.append(cow_id)

        # Extract iStateVars
        iStateVars = [row[col] for col in df.columns if col.startswith('iStateVar_')]
        iStateVars_list.append(iStateVars)

        # Extract parameters
        parameters_dict = {col.replace('param_', ''): row[col] for col in df.columns if col.startswith('param_')}
        parameters_dict_list.append(parameters_dict)

    # Create a new DataFrame
    wrangled_data = pd.DataFrame({
        'cow_ID': cow_ids,
        'iStateVars': iStateVars_list,
        'parameters': parameters_dict_list
    })

    return wrangled_data

cow_data_wranlged = cow_wrangler(cow_data_raw)

# Initalize herd
pregdate_values = {
    'pregdate_1': 120,
    'pregdate_2': 141,
    'pregdate_3': 162,
    'pregdate_4': 183,
    'pregdate_5': 204
}

output = ['t', 'dijkstra_milk', 'NEmilk', 'milk', 'DMI', 'E', 'BW', 'BFat', 'ER', 'IOFC', 'milkcumul', 'DMIcumul', 'IOFCcumul'] # Model outputs

herd = moo.Herd(outputs=output, model_functions=functions_to_include, scenario_dict=pregdate_values)

# Load cows from dataframe
herd.import_dataframe(df=cow_data_wranlged)


All cows have been added


In [8]:
herd.run_all_models(0,
                    250,
                    0.01,
                    7,
                    0)

Begin executing models...
Running Model....
         t  dijkstra_milk     NEmilk       milk        DMI         E  \
0     0.00      14.152526   9.553224  12.754638   8.701276  0.378823   
1     6.99      18.592837  11.267327  15.043160  10.606871  0.413388   
2    13.99      22.484074  14.452919  19.296287  12.848141  0.458688   
3    20.99      25.616585  17.268687  23.055657  14.912767  0.498827   
4    27.99      27.967443  19.606277  26.176604  16.729624  0.534491   
5    34.99      29.616951  21.448153  28.635718  18.267823  0.566006   
6    41.99      30.686991  22.831983  30.483288  19.528943  0.593644   
7    48.99      31.303405  23.821411  31.804287  20.535162  0.617713   
8    55.99      31.577763  24.486708  32.692534  21.318764  0.638559   
9    62.99      31.601204  24.894026  33.236350  21.914789  0.656552   
10   69.99      31.444560  25.100630  33.512190  22.356693  0.672057   
11   76.99      31.161127  25.153645  33.582970  22.674228  0.685421   
12   83.99      30.7

In [3]:
####################
# Step 3: Run model with 5 values for pregdate
####################

herd.run_all_scenarios(0,
                       500,
                       0.01,
                       7,
                       0)





# class scenario:
#     def __init__(self, parameter_list):
#         self.parameter_list = parameter_list


# TODO
# 1. Create a list with 5 pregdate values, stored at herd level
# 2. Write a modified run_all_models that runs all scenarios
    # Iterate over cows
    # For a cow iterate through Scenario
    # Execute model with each different parameters


Running Model....
This is the scenario_param: 120
         t  dijkstra_milk     NEmilk       milk        DMI         E  \
0     0.00      14.152526   9.553224  12.754638   8.701276  0.378823   
1     6.99      18.592837  11.267327  15.043160  10.606871  0.413388   
2    13.99      22.484074  14.452919  19.296287  12.848141  0.458688   
3    20.99      25.616585  17.268687  23.055657  14.912767  0.498827   
4    27.99      27.967443  19.606277  26.176604  16.729624  0.534491   
..     ...            ...        ...        ...        ...       ...   
67  468.99      10.837858   9.906662  13.226518  20.789191  0.853472   
68  475.99      10.630480   9.719724  12.976934  20.795248  0.853878   
69  482.99      10.427071   9.536164  12.731861  20.802705  0.854261   
70  489.99      10.227553   9.355926  12.491224  20.811523  0.854620   
71  496.99      10.031853   9.178954  12.254946  20.821661  0.854957   

            BW        BFat           ER       IOFC     milkcumul  \
0   650.000000  1

In [4]:
all_dataframes = []

for cow in herd.list_of_cows:
    cow_dataframes = cow.results  # Assuming 'results' is a list containing 5 dataframes
    all_dataframes.extend(cow_dataframes)


In [5]:
print(cow_dataframes[0])

df_1 = cow_dataframes[0]
df_2 = cow_dataframes[1]
df_3 = cow_dataframes[2]
df_4 = cow_dataframes[3]
df_5 = cow_dataframes[4]


         t  dijkstra_milk     NEmilk       milk        DMI         E  \
0     0.00      26.175011  17.668630  23.589626  11.064069  0.358281   
1     6.99      31.906827  18.438046  24.616884  13.050822  0.380144   
2    13.99      35.674453  22.081032  29.480684  15.688198  0.429102   
3    20.99      37.850032  24.784368  33.089944  17.915991  0.473894   
4    27.99      38.906555  26.677091  35.616944  19.719764  0.513999   
..     ...            ...        ...        ...        ...       ...   
67  468.99      11.303676  10.278382  13.722806  20.700758  0.845610   
68  475.99      11.071099  10.069831  13.444368  20.692258  0.846044   
69  482.99      10.843307   9.865348  13.171360  20.685551  0.846454   
70  489.99      10.620202   9.664858  12.903683  20.680584  0.846839   
71  496.99      10.401687   9.468287  12.641238  20.677306  0.847200   

            BW        BFat           ER       IOFC     milkcumul  \
0   695.000000  125.100000  1125.900000  19.412836      0.000000   